In [ ]:

def enrich_tf_local(net, adata_bulk, tf_all):
    net = net.pivot(index='source', columns='target', values='weight').fillna(0)
    net = net[[g for g in adata_bulk.var_names if g in net.columns]]
    tfs_present = np.intersect1d(net.index, tf_all)
    net = net[net.index.isin(tfs_present)]
    print('ratio of porosity: ', (net==0).sum().sum()/net.size)
    # - subset the adata
    adata_bulk = adata_bulk[:, adata_bulk.var_names.isin(net.columns)]
    # - enrich tfs 
    mat = adata_bulk.X.todense().T
    
    tf_acts = np.dot(net, mat)
    # - format
    tf_acts = pd.DataFrame(tf_acts, index=net.index, columns=adata_bulk.obs.index)
    tf_acts = tf_acts.reset_index().melt(id_vars='source', var_name='sample', value_name='activity')
    tf_acts = tf_acts.set_index('sample').merge(adata_bulk.obs[['cell_type', 'donor_id', 'cell_count', 'age']], left_index=True, right_index=True).reset_index(drop=False)
    print(f"net: {net.shape}, mat: {mat.shape}, source: {tf_acts['source'].nunique()}")
    # Calculate ranks within each sample
    tf_acts['rank'] = tf_acts.groupby('sample')['activity'].transform(lambda x: x.abs().rank(method='dense', ascending=False)) 

    return tf_acts



def enrich_tfs(adata_sample, net, tf_all):
    # import decoupler
    # from scipy.stats import zscore

    # - pseudobulk cell type-donor
    # sys.path.insert(0, '../')
    # from task_grn_inference.src.process_data.perturbation.opsca.script import sum_by

    
    # -enrich TFs
    net = net[net['source'].isin(tf_all)]

    if False:
        mat = pd.DataFrame(
            data=adata_bulk.X.todense(),  
            columns=adata_bulk.var_names,  
            index=adata_bulk.obs.index  
        )

        tf_acts, tf_pvals = decoupler.run_ulm(mat, net, source='source', target='target', weight='weight', use_raw=False)
        # - formatize
        tf_acts = tf_acts.reset_index().melt(id_vars='index', var_name='source', value_name='activity')
        obs = adata_bulk.obs[['cell_type', 'donor_id', 'cell_count', 'age']]
        obs = obs.reset_index()
        
        tf_acts['index'] = tf_acts['index'].astype(str)
        obs['index'] = obs['index'].astype(str)
        tf_acts = tf_acts.merge(obs, on='index', how='left').drop('index', axis=1)
        assert tf_acts.shape[0]==tf_acts.shape[0]
    else:
        tf_acts = enrich_tf_local(net, adata_bulk, tf_all)

    return tf_acts


net_MONO_all_agegroups_all_batches.csv


In [8]:
!ls /vol/projects/jnourisa/

adata_all_bulk.h5ad	    adata_data7_raw.h5ad
adata_all.h5ad		    adata_pbmc_ageing.h5ad
adata_all_processed.h5ad    adata_test.h5ad
adata_all_raw.h5ad	    alis_preprocessing.ipynb
adata_data1_processed.h5ad  all_only_male_False_downsample_False.h5ad
adata_data1_raw.h5ad	    all_only_male_True_downsample_False.h5ad
adata_data7_processed.h5ad  pbmc_ageing_only_male_True_downsample_False.h5ad


In [ ]:
import anndata as ad
import pandas as pd
import sys
import numpy as np



par = {
    'adata_bulk': "/vol/projects/jnourisa/adata_data1_raw.h5ad",
    'nets': ["../output/grns/data1/net_MONO_all_agegroups_all_batches.csv"],
    'cell_types': ['MONO'],
    'n_cells_t': 10
    'tf_all': "../../task_grn_inference/resources/grn_benchmark/prior/tf_all.csv",

}


adata_bulk = ad.read_h5ad(par['adata_bulk'], backed='r')
tf_all = np.loadtxt(par['tf_all'], dtype=str)

tf_acts_store = []
for i, cell_type in enumerate(par['cell_types']):
    mask_sample = adata_bulk.obs['cell_type']==cell_type 
    net = par['nets'][i]
    net = pd.read_csv(net)
    adata_sample = adata_bulk[mask_sample, :].to_memory()   
    adata_sample = adata_sample[adata_sample.obs['cell_count']>par['n_cells_t']]
    aa

    tf_acts = enrich_tfs(adata_sample, net, tf_all)
    tf_acts['cell_type'] = cell_type

    tf_acts_store.append(tf_acts)

tf_acts_all = pd.concat(tf_acts_store)

KeyError: 'cell_type'

In [6]:
!ls ../../task_grn_inference/resources/grn_benchmark/prior

op				     skeleton.csv
regulators_consensus_adamson.json    tf_all.csv
regulators_consensus_nakatake.json   ws_consensus_adamson.csv
regulators_consensus_norman.json     ws_consensus_norman.csv
regulators_consensus_op.json	     ws_distance_background_adamson.csv
regulators_consensus_replogle2.json  ws_distance_background_norman.csv


In [11]:

    
    
    
    for age_group in par['age_groups']: 
        mask_age = (adata.obs['age_group']==age_group)

        if par['only_male']:
            mask_gender = (adata.obs['sex']=='M')
            mask_sample = mask_age&mask_gender&mask_celltype 
        else:
            mask_sample = mask_age
        print('Number of cells: ', mask_sample.sum())
        adata_sample = adata[mask_sample, :].to_memory()

        net = pd.read_csv(os.path.abspath(f"output/grns/{dataset}/net_{cell_type}_{age_group}_all_batches.csv"))
        
        tf_acts = enrich_tfs(adata_sample, net, tf_all, par)
        tf_acts['age_group'] = age_group
        tf_acts['dataset'] = dataset
        # tf_acts['dataset'] = adata_sample.ob
        tf_acts_store.append(tf_acts)
    
        tf_acts_all = pd.concat(tf_acts_store)
    tf_acts_all.to_csv(par['save_file'])

op				     skeleton.csv
regulators_consensus_adamson.json    tf_all.csv
regulators_consensus_nakatake.json   ws_consensus_adamson.csv
regulators_consensus_norman.json     ws_consensus_norman.csv
regulators_consensus_op.json	     ws_distance_background_adamson.csv
regulators_consensus_replogle2.json  ws_distance_background_norman.csv
